# High-Impedance Ring Resonator Analysis

This notebook documents the analysis and plotting workflow for measurement data from high-impedance ring resonators.

## Overview
It includes data loading, preprocessing, fitting, and visualization steps used to study resonator behavior and extract relevant parameters from measured datasets.

## Contents
- loading qkit measurement data
- frequency, phase, and amplitude analysis
- circle fitting
- spectral analysis
- plotting of raw and processed results
- comparison of selected measurements.

## Workflow
1. import required libraries  
2. configure qkit and dataset paths  
3. load measurement files  
4. process and analyze data  
5. visualize results  
6. compare and summarize observations

# Imports

In [ ]:
# ============================================================
# 1. Imports
# Standard library
# ============================================================
import os
import re
from time import sleep

# ============================================================
# 2. Scientific Python stack
# ============================================================
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import scipy.optimize
from scipy.constants import physical_constants
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from scipy.signal import periodogram
from scipy.stats import norm

# ============================================================
# 3. QKIT imports
# ============================================================
# ============================================================
# 2. QKIT configuration
# ============================================================

import qkit
from qkit.analysis.circle_fit.circle_fit_2019 import circuit
from qkit.analysis.circle_fit.circle_fit_2019.circuit import reflection_port
from qkit.gui.notebook.Progress_Bar import Progress_Bar
from qkit.measure.spectroscopy.spectroscopy import spectrum
from qkit.storage.store import Data

# ============================================================
# 4. Physical constants
# ============================================================
k_B = 1.380649e-23   # Boltzmann constant [J/K]
h = 6.62607015e-34   # Planck constant [J*s]
hbar = 1.0545718e-34 # Reduced Planck constant [J*s]

# QKIT Configuration

In [ ]:
# Data source configuration
DATA_DIR = r"C:\qkit\qkit\data\"
RUN_ID = ""
USER = "Mahya"

In [ ]:
# Measurement / sample configuration
CHIP_NAME = "GRRL75_2T1"

# Apply qkit configuration
qkit.cfg["datadir"] = DATA_DIR
qkit.cfg["run_id"] = RUN_ID
qkit.cfg["user"] = USER

# Start qkit framework
qkit.start()

## Functions

In [ ]:
def load_h5(date, foldername):
    path = "E:/{}/{}/{}.h5".format(date, foldername, foldername)
    return Data(path)

def load_txt(date, foldername):
    path = "D:/{}/{}/{}.txt".format(date, foldername, foldername)
    return np.loadtxt(path)

# Plotting Setting

In [ ]:
PLOT_STYLE = {
    "text.usetex": True,
    "text.latex.preamble": r"""
        \usepackage{amsmath}
        \usepackage{siunitx}
        \sisetup{detect-all, separate-uncertainty}
    """,
    "font.family": "serif",
    "font.serif": [],
    "font.size": 18,
    "font.weight": "bold",
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "axes.labelweight": "bold",
    "axes.linewidth": 1.5,
    "legend.fontsize": 18,
    "legend.frameon": False,
    "legend.loc": "best",
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.minor.width": 1,
    "ytick.minor.width": 1,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "figure.figsize": (7.5, 5),
    "figure.autolayout": True,
    "figure.facecolor": "white",
    "axes.edgecolor": "black",
    "lines.linewidth": 2.5,
}

mpl.rcParams.update(PLOT_STYLE)

# Circle-Fit Analysis 

In [ ]:
# Initialize your data list outside the loop
data = []

datasets= ['']

for UUID in datasets:
    try:
        # Retrieve data from file
        h5file = Data(qkit.fid.get(UUID))
        freq = np.array(h5file.data.frequency)
        pha = np.array(h5file.data.phase)
        amp = np.array(h5file.data.amplitude)
        h5file.close_file()

        # Fit the circuit model
        cfit = circuit.reflection_port(freq, z_data_raw=amp * np.exp(1j * pha))
        cfit.autofit()

        # Extract and print fitting parameters
        fr = cfit.fr
        Ql = cfit.Ql
        Qc = cfit.Qc
        Qi = 1 / (1 / Ql - 1 / Qc)
        power = -15
        attenuation = 96
        nbar = 4 * 10 ** ((power  - 30 - attenuation) / 10) * Ql ** 2 / (hbar * (2 * np.pi * fr) ** 2 * Qc)

        print(f"Dataset {UUID}:")
        print(f"fr = {fr*1.e-9:.4f} GHz")
        print(f"Ql = {Ql:.0f}")
        print(f"Qc = {Qc:.0f}")
        print(f"Qi = {Qi:.0f}")
        print(f"k  = {fr / Ql *1.e-6:.3f} MHz")
        print(f"nbar  = {nbar}")
        
        #save_path = f"C:\\PhD_project\\measurements\\ring resonators\\2023-08-11_GRRL75_2T1\\CircleFit_plot_{UUID}.png"
        #plt.savefig(save_path, dpi=300, bbox_inches='tight')
        # Plot the results
        cfit.plotall()

        plt.close('all')
        # Append the data for this UUID to our list
        data.append({
            'UUID': UUID,
            'fr (GHz)': fr * 1e-9,
            'Ql': Ql,
            'Qc': Qc,
            'Qi': Qi,
            'k (MHz)': fr / Ql *1.e-6
        })

    except Exception as e:  # This catches any exceptions that occur in the try block
        print(f"An error occurred while processing {UUID}: {e}")

In [ ]:
# After the loop, convert the list of data into a pandas DataFrame
df = pd.DataFrame(data)

# Print the DataFrame to see the table
print(df)

# Optionally, save the DataFrame to a CSV file
df.to_csv('resonator_parameters.csv', index=False)

# Phase Response

## Phase-Plot Configuration


In [ ]:
DATASETS = ["RZY6G3", "S8SK59", "S2Y1F2"]
PLOT_COLORS = ["#00c0f3", "#00A76D", "#9f58db"]

# Frequency windows for each dataset [GHz]
X_LIMITS = {
    "RZY6G3": (5.955, 5.987),
    "S8SK59": (5.780, 5.875),
    "S2Y1F2": (7.015, 7.110),
}

FIG_SIZE = (1.4, 0.75)
OUTPUT_DPI = 300   # 300 is usually enough for publication export
SAVE_DIR = r"C:\PhD_project\Paper\Hight_Impedance_Resonators\figs\3"

## Generate Phase Plots

In [ ]:
for dataset_id, color in zip(DATASETS, PLOT_COLORS):
    # Create figure
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    # Load data
    h5file = Data(qkit.fid.get(dataset_id))
    freq_ghz = h5file.data.frequency[:] / 1e9
    phase = h5file.data.phase[:]
    h5file.close_file()

    # Plot phase response
    ax.plot(freq_ghz, phase, linewidth=1, color=color)

    # Axis labels
    ax.set_ylabel(r'$\arg(S_{11}) \, \mathrm{(rad)}$', fontsize=8, labelpad=2, color='black')
    ax.set_xlabel(r'$\mathit{f} \, \mathrm{(GHz)}$', fontsize=8, labelpad=2, color='black')

    # Tick formatting
    ax.tick_params(axis='both', which='major', length=2, labelsize=8, width=1, colors='black')

    # X-axis range and ticks
    x_start, x_end = X_LIMITS[dataset_id]
    x_ticks = np.linspace(x_start, x_end, 3)

    ax.set_xlim(x_start - 0.002, x_end + 0.002)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f"{x:.2f}" for x in x_ticks])

    # Layout
    fig.subplots_adjust(left=0.25, right=0.95, top=0.85, bottom=0.02)

    # Save figure
    save_path = os.path.join(SAVE_DIR, f"Phase_plot_{dataset_id}.png")
    fig.savefig(save_path, dpi=OUTPUT_DPI, bbox_inches="tight")

    plt.show()
    plt.close(fig)

# Internal Quality Factor vs Photon Number

## Qi vs Photon Number Configuration

In [ ]:
DATASETS = [""]  # Replace with valid dataset UUIDs

UUID_TO_STYLE = {
    "": {
        "color": "#9f58db",
        "marker": "s",
        "label": r"$Q_c = 150k$",
        "linestyle": "--",
        "linewidth": 1,
        "markersize": 3,
        "attenuation_db": 96,
        "power_range_dbm": (-80, 7),
    },
}

FIG_SIZE = (4.2, 2.1)
X_LIMITS = (1e-1, 1.2e3)
Y_LIMITS = (8e4, 7e6)
FIT_QUALITY_THRESHOLD = 0.5

## Qi vs Photon Number Analysis and Plotting

In [ ]:
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.set_position([0.15, 0.15, 0.8, 0.8])

ax.set_xscale("log")
ax.set_yscale("log")

for uuid in DATASETS:
    style = UUID_TO_STYLE[uuid]

    # Load dataset
    h5file = Data(qkit.fid.get(uuid))
    freq = h5file.data.frequency[:]
    amplitude = h5file.data.amplitude[:]
    phase = h5file.data.phase[:]
    power_data = h5file.data.power[:]
    h5file.close_file()

    # Convert amplitude and phase to complex response
    z_data = amplitude * np.exp(1j * phase)

    # Restrict analysis to selected power range
    power_min, power_max = style["power_range_dbm"]
    valid_power_mask = (power_data >= power_min) & (power_data <= power_max)

    selected_powers = power_data[valid_power_mask]
    selected_z_data = z_data[valid_power_mask]

    # Containers for fit results
    fitted_qi = []
    fitted_qi_err = []
    fitted_ql = []
    fitted_qc = []
    fitted_qc_abs_err = []
    fitted_fr = []
    fitted_fr_err = []
    fitted_powers = []
    fitted_chi_square = []

    # Fit each power trace
    for i, power_dbm in enumerate(selected_powers):
        cf = circuit.reflection_port(freq, selected_z_data[i])
        cf.autofit()

        qi = cf.fitresults["Qi"]
        qi_err = cf.fitresults["Qi_err"]

        if abs(qi_err / qi) < FIT_QUALITY_THRESHOLD:
            fitted_powers.append(power_dbm)
            fitted_qc.append(cf.fitresults["Qc"])
            fitted_ql.append(cf.fitresults["Ql"])
            fitted_fr.append(cf.fitresults["fr"])
            fitted_qi.append(qi)
            fitted_fr_err.append(cf.fitresults["fr_err"])
            fitted_qc_abs_err.append(cf.fitresults["absQc_err"])
            fitted_qi_err.append(qi_err)
            fitted_chi_square.append(cf.fitresults["chi_square"])

    # Convert to arrays
    fitted_qi = np.array(fitted_qi)
    fitted_qi_err = np.array(fitted_qi_err)
    fitted_ql = np.array(fitted_ql)
    fitted_qc = np.array(fitted_qc)
    fitted_qc_abs_err = np.array(fitted_qc_abs_err)
    fitted_fr = np.array(fitted_fr)
    fitted_fr_err = np.array(fitted_fr_err)
    fitted_powers = np.array(fitted_powers)
    fitted_chi_square = np.array(fitted_chi_square)

    # Compute average photon number
    attenuation_db = style["attenuation_db"]
    nbar = (
        4
        * 10 ** ((fitted_powers - 30 - attenuation_db) / 10)
        * fitted_ql**2
        / (hbar * (2 * np.pi * fitted_fr) ** 2 * fitted_qc)
    )

    # Estimate uncertainty band for Qi
    coupling_ratio = fitted_ql / fitted_qc
    b = 10 ** (-20 / 20)
    b_tilde = b / (1 - b)
    coupling_ratio_err = coupling_ratio * b_tilde

    qi_min = fitted_ql / (1 - (coupling_ratio - coupling_ratio_err))
    qi_max = fitted_ql / (1 - (coupling_ratio + coupling_ratio_err))

    # Restrict displayed photon-number range
    display_mask = (nbar > 1e-1) & (nbar < 1e7)

    print(f"Dataset: {uuid}")
    print("fr:", fitted_fr)
    print("Qc:", fitted_qc)

    # Plot uncertainty band and fitted Qi
    ax.fill_between(
        nbar[display_mask],
        qi_min[display_mask],
        qi_max[display_mask],
        alpha=0.25,
        color=style["color"],
    )

    ax.plot(
        nbar[display_mask],
        fitted_qi[display_mask],
        style["marker"],
        color=style["color"],
        linestyle=style["linestyle"],
        linewidth=style["linewidth"],
        markersize=style["markersize"],
        label=style["label"],
    )

# Axis labels and formatting
ax.set_ylabel(r"$Q_i$", fontsize=8, labelpad=3, color="black")
ax.set_xlabel(r"$\mathit{\bar{n}}$", fontsize=8, labelpad=2, color="black")

ax.tick_params(axis="both", which="major", length=2, labelsize=8, pad=1)
ax.tick_params(axis="both", which="minor", length=1.3, width=0.7)

ax.set_xlim(*X_LIMITS)
ax.set_ylim(*Y_LIMITS)

ax.legend(
    loc="upper left",
    fontsize=8,
    handletextpad=0.5,
    labelspacing=0.33,
    frameon=False,
    bbox_to_anchor=(0, 1.05),
)

plt.show()

# Kerr Shift 

## Fitting Functions

In [ ]:
def _linefunc(x,k,d):
    return k*x +d

## Kerr-shift Analysis

In [ ]:
DATASETS = [""]  # Replace with valid dataset UUID(s)
FIT_QUALITY_THRESHOLD = 2000

kerr_results = []

for uuid in DATASETS:
    # Load dataset
    h5file = Data(qkit.fid.get(uuid))
    freq = np.array(h5file.data.frequency)
    amplitude = np.array(h5file.data.amplitude)
    phase = np.array(h5file.data.phase)
    power_data = np.array(h5file.data.power)
    h5file.close_file()

    z_data = amplitude * np.exp(1j * phase)

    # ------------------------------------------------------------------
    # Full-range resonance-frequency extraction
    # ------------------------------------------------------------------
    full_frs = []
    full_frs_err = []

    for i, power_dbm in enumerate(power_data):
        cf = circuit.reflection_port(freq, z_data[i])
        cf.autofit()
        full_frs.append(cf.fitresults["fr"])
        full_frs_err.append(cf.fitresults["fr_err"])

    full_frs = np.array(full_frs)
    full_frs_err = np.array(full_frs_err)

    # ------------------------------------------------------------------
    # Restricted-range analysis for Kerr fit
    # ------------------------------------------------------------------
    style = uuid_to_style[uuid]
    power_min, power_max = style["power_range"]

    valid_power_mask = (power_data >= power_min) & (power_data <= power_max)
    filtered_powers = power_data[valid_power_mask]
    filtered_z_data = z_data[valid_power_mask]

    ql_values = []
    qc_values = []
    fr_values = []
    fr_err_values = []
    qi_values = []
    qi_err_values = []
    qc_abs_err_values = []
    chi_square_values = []

    for i, power_dbm in enumerate(filtered_powers):
        cf = circuit.reflection_port(freq, filtered_z_data[i])
        cf.autofit()

        qi = cf.fitresults["Qi"]
        qi_err = cf.fitresults["Qi_err"]

        if abs(qi_err / qi) < FIT_QUALITY_THRESHOLD:
            qc_values.append(cf.fitresults["Qc"])
            ql_values.append(cf.fitresults["Ql"])
            fr_values.append(cf.fitresults["fr"])
            qi_values.append(qi)
            fr_err_values.append(cf.fitresults["fr_err"])
            qc_abs_err_values.append(cf.fitresults["absQc_err"])
            qi_err_values.append(qi_err)
            chi_square_values.append(cf.fitresults["chi_square"])

    # Convert to arrays
    ql_values = np.array(ql_values)
    qc_values = np.array(qc_values)
    fr_values = np.array(fr_values)
    fr_err_values = np.array(fr_err_values)
    qi_values = np.array(qi_values)
    qi_err_values = np.array(qi_err_values)
    qc_abs_err_values = np.array(qc_abs_err_values)
    chi_square_values = np.array(chi_square_values)

    print(f"Dataset: {uuid}")
    print("fr:", fr_values)

    # Photon-number estimate
    nbar = (
        4
        * 10 ** ((filtered_powers - 30 - style["attenuation"]) / 10)
        * ql_values**2
        / (hbar * (2 * np.pi * fr_values) ** 2 * qc_values)
    )

    # Linear fit of resonance shift
    popt, pcov = scipy.optimize.curve_fit(_linefunc, nbar, fr_values - fr_values[0])
    fitted_fr_shift = _linefunc(nbar, *popt)

    # Store results for plotting
    kerr_results.append(
        {
            "uuid": uuid,
            "style": style,
            "nbar": nbar,
            "fr_values": fr_values,
            "fr_shift_khz": (fr_values - fr_values[0]) / 1000,
            "fitted_fr_shift_khz": fitted_fr_shift / 1000,
            "full_frs": full_frs,
            "full_frs_err": full_frs_err,
            "fit_parameters": popt,
            "fit_covariance": pcov,
        }
    )

## Plot Kerr Shift vs Photon Number

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 2.1))

for result in kerr_results:
    style = result["style"]

    ax.scatter(
        result["nbar"],
        result["fr_shift_khz"],
        s=style["markersize"],
        edgecolor=style["color"],
        facecolors="none",
        marker=style["marker"],
        label=style["label"],
        linewidth=0.5,
    )

    ax.plot(
        result["nbar"],
        result["fitted_fr_shift_khz"],
        linestyle=style["linestyle"],
        color=style["color"],
        linewidth=style["linewidth"],
    )

ax.set_xlabel(r"$\mathit{\bar{n}}$", fontsize=8, labelpad=2)
ax.set_ylabel(r"$\Delta f \, \mathrm{(kHz)}$", fontsize=8, labelpad=2)

ax.tick_params(axis="both", which="major", length=2, labelsize=8, pad=1)
ax.set_ylim(-650, None)

ax.legend(frameon=False, fontsize=8)

print("Axis position:", ax.get_position())
plt.show()

# TimeTrace Measurements


## Helper Functions for Frequency Reconstruction and Noise Analysis

In [ ]:
def _fake_trace(x):
    """Placeholder helper function. Currently unused."""
    return


def f_from_sk1(s11_data, f_data, Ql, Qc, phi, a, alpha, delay, k):
    """
    Reconstruct corrected frequency values from complex reflection data.

    Parameters
    ----------
    s11_data : array-like
        Complex reflection data.
    f_data : array-like
        Frequency axis.
    Ql, Qc, phi, a, alpha, delay, k : float
        Circle-fit parameters.

    Returns
    -------
    ff : ndarray
        Corrected frequency values.
    """
    # Complex scaling factor including amplitude, phase, and delay
    E = a * np.exp(1j * alpha) * np.exp(-2j * np.pi * f_data * delay)

    # Fit-dependent scaling factors
    A = (2 / k) * (Ql / Qc) * np.exp(1j * phi)
    B = 2j * Ql

    # Frequency correction term
    x = np.real((1 / B) * ((E * A / (E - s11_data)) - 1))

    # Corrected frequency
    ff = f_data / (x + 1)

    return ff


def use_circlefit(freq, z, n_steps):
    """
    Perform circle fitting and generate a resampled trace around resonance.

    Parameters
    ----------
    freq : array-like
        Frequency axis.
    z : array-like
        Complex resonator response.
    n_steps : int
        Number of sampling points around resonance.

    Returns
    -------
    f_sampling : ndarray
        Resampled frequency axis around the fitted resonance.
    z_sampling : ndarray
        Simulated complex response on the resampled frequency axis.
    """
    cf = reflection_port(freq, z)
    cf.fit_delay_max_iterations = 100
    cf.autofit()

    fr = cf.fitresults["fr"]
    k = cf.fitresults["fr"] / cf.fitresults["Ql"]

    f_sampling = np.linspace(fr - k / 2, fr + k / 2, n_steps)
    z_sampling = cf.Sij(
        f_sampling,
        cf.fitresults["fr"],
        cf.fitresults["Ql"],
        cf.fitresults["Qc"],
        cf.fitresults["phi"],
        cf.fitresults["a"],
        cf.fitresults["alpha"],
        cf.fitresults["delay"],
    )

    return f_sampling, z_sampling


def PSD_avg(ft, t_per_trace):
    """
    Compute the average power spectral density (PSD) over multiple traces.

    Parameters
    ----------
    ft : list of array-like
        Frequency traces.
    t_per_trace : float
        Time per trace.

    Returns
    -------
    xf : ndarray
        Frequency axis for the PSD.
    psd : ndarray
        Averaged power spectral density.
    ft : list
        Original input traces.
    """
    for j, f_t in enumerate(ft):
        if j == 0:
            xf, psd = periodogram(f_t, 1 / t_per_trace)
        else:
            psd += periodogram(f_t, 1 / t_per_trace)[1]

    psd = psd / len(ft)
    return xf, psd, ft


def NSD_avg(ft, t_per_trace):
    """
    Compute the average noise spectral density (NSD) from multiple traces.
    """
    xf, psd, ft = PSD_avg(ft, t_per_trace)
    nsd = np.sqrt(psd)
    return xf, nsd, ft


def derivative(array):
    """
    Compute a discrete first-difference derivative of an array.
    """
    der = [0]
    for i in range(len(array) - 1):
        der.append(array[i + 1] - array[i])
    return -np.asarray(der)


def threshold_calc(f_ts):
    """
    Compute the derivative trace and standard deviation threshold parameter.
    """
    der = derivative(f_ts)
    sigma = np.std(f_ts)
    return der, sigma


def refine_traces(der, thres):
    """
    Flag traces that exceed a derivative threshold.

    Returns
    -------
    c : int
        0 if the trace is accepted, 1 if it is flagged.
    """
    c = 0
    for value in der:
        if abs(value) > thres:
            c = 1
    return c


def PSD_calc(f_ts, derivatives, thres, t_per_trace):
    """
    Filter traces based on derivative threshold and compute averaged PSD.

    Parameters
    ----------
    f_ts : list of array-like
        Frequency traces.
    derivatives : list of array-like
        Derivative traces corresponding to f_ts.
    thres : float
        Threshold used for trace rejection.
    t_per_trace : float
        Time per trace.

    Returns
    -------
    f : ndarray
        Frequency axis.
    psd_avg : ndarray
        Averaged PSD of accepted traces.
    ft : list
        Accepted traces.
    delete : list
        Rejected traces.
    """
    ft = []
    delete = []

    for i, ff in enumerate(f_ts):
        c = refine_traces(derivatives[i], thres)
        print(c)

        if c == 0:
            ft.append(ff)
        else:
            delete.append(ff)

    print(len(ft))
    f, psd_avg, ft = PSD_avg(ft, t_per_trace)
    return f, psd_avg, ft, delete


def inverted_freq(f_sampling, z_sampling, z_ts):
    """
    Convert complex time traces into frequency traces using nearest-neighbor matching.

    Parameters
    ----------
    f_sampling : ndarray
        Reference sampled frequency axis.
    z_sampling : ndarray
        Reference sampled complex response.
    z_ts : list of array-like
        Complex time traces.

    Returns
    -------
    f_ts : list
        Reconstructed frequency traces.
    delete : list
        Indices of traces rejected for going out of the valid sampling range.
    """
    f_ts = []
    delete = []

    for i, z_tt in enumerate(z_ts):
        f_t = [f_sampling[np.argmin(np.abs(z_sampling - z_t))] for z_t in z_tt]
        f_t = np.array(f_t)

        if np.any(f_t < f_sampling[10]) or np.any(f_t > f_sampling[-10]):
            print("🚨🚨👮 Sus")
            delete.append(i)
            continue

        f_ts.append(f_t)

    return f_ts, delete

## Circle Comparison: Full-Circle and Zero-Span Measurements

In [ ]:
def load_data_from_uuid(uuid):
    """
    Load frequency, complex response, and optional time data from a qkit dataset.

    Parameters
    ----------
    uuid : str
        Dataset UUID.

    Returns
    -------
    freq : ndarray
        Frequency axis.
    z_data : ndarray
        Complex resonator response.
    ts : ndarray or None
        Time axis if available, otherwise None.
    """
    qkit.fid.update_all()

    h5file = Data(qkit.fid.get(uuid))
    freq = np.array(h5file["entry/data0/frequency"])
    amplitude = np.array(h5file["entry/data0/amplitude"])
    phase = np.array(h5file["entry/data0/phase"])

    try:
        ts = np.array(h5file["entry/data0/time"])
    except KeyError:
        print("Warning: 'time' field not found in the dataset.")
        ts = None

    h5file.close_file()

    z_data = amplitude * np.exp(1j * np.unwrap(phase))
    return freq, z_data, ts


def z_spline(frequency_axis, i_prediction, q_prediction):
    """
    Evaluate the spline-based complex resonator response.
    """
    i_values = splev(frequency_axis, i_prediction)
    q_values = splev(frequency_axis, q_prediction)
    return q_values + 1j * i_values


def plot_circle_fit(
    title,
    z_data,
    z_interpolated,
    trace_index=None,
    ylim=None,
    xlim=None,
    marker_size=3,
    fit_line_width=1,
    fit_color="blue",
    point_color="red",
):
    """
    Plot interpolated circle fit together with measured complex data.
    """
    plt.figure()
    plt.title(title)

    plt.plot(
        z_interpolated.real,
        z_interpolated.imag,
        color=fit_color,
        linewidth=fit_line_width,
    )

    if trace_index is not None:
        plt.plot(
            z_data[trace_index].real,
            z_data[trace_index].imag,
            "o",
            ms=marker_size,
            color=point_color,
        )
    else:
        z_data_flat = z_data.flatten()
        plt.plot(
            z_data_flat.real,
            z_data_flat.imag,
            "o",
            ms=marker_size,
            color=point_color,
        )

    plt.gca().set_aspect("equal")

    if xlim is not None:
        plt.xlim(xlim)
    if ylim is not None:
        plt.ylim(ylim)

    plt.show()

## Dataset Selection for Circle Comparison

In [ ]:
UUID_PAIRS = [
    ("S90JR6", "S90JS0"),
]

SPLINE_SMOOTHING = 3.5e-6
N_INTERPOLATION_POINTS = 1000

## Compare Full-Circle and Zero-Span Measurements

In [ ]:
for full_circle_uuid, zero_span_uuid in UUID_PAIRS:
    # --------------------------------------------------------
    # Full-circle measurement
    # --------------------------------------------------------
    freq_full, z_full, _ = load_data_from_uuid(full_circle_uuid)

    i_prediction_full = splrep(freq_full, z_full.imag, s=SPLINE_SMOOTHING)
    q_prediction_full = splrep(freq_full, z_full.real, s=SPLINE_SMOOTHING)

    f_interpolated = np.linspace(freq_full.min(), freq_full.max(), N_INTERPOLATION_POINTS)
    z_interpolated_full = z_spline(f_interpolated, i_prediction_full, q_prediction_full)

    plot_circle_fit(
        title=f"Full Circle Measurement — {full_circle_uuid}",
        z_data=z_full,
        z_interpolated=z_interpolated_full,
        fit_color="green",
        point_color="orange",
        fit_line_width=2,
    )

    # --------------------------------------------------------
    # Zero-span measurement
    # --------------------------------------------------------
    _, z_zero_span, _ = load_data_from_uuid(zero_span_uuid)

    plot_circle_fit(
        title=f"Zero Span Measurement — {zero_span_uuid}",
        z_data=z_zero_span,
        z_interpolated=z_interpolated_full,
        trace_index=None,
        marker_size=1,
        fit_color="purple",
        point_color="cyan",
        fit_line_width=0.5,
    )

## Power Spectral Density of Frequency-Time Traces

In [ ]:
PROCESSED_DATA_FILE = "Time_trace_S8ZU7J_S2XXMP_S0BI8H_S9SL0J.npz"

# Load processed resonator time-trace data
resonator_data = np.load(PROCESSED_DATA_FILE, allow_pickle=True)


# Compute and plot averaged PSD for each dataset
fig, ax = plt.subplots(figsize=(15, 10))

for uuid in resonator_data:
    dataset = resonator_data[uuid].item()

    f_ts = dataset["frequency_time_series"]
    t_per_trace = np.mean(np.diff(dataset["time_series"]))

    # Compute averaged PSD
    fft_f, psd_avg, _ = PSD_avg(f_ts, t_per_trace)

    # Plot PSD
    ax.plot(
        fft_f,
        psd_avg,
        marker="o",
        ms=1,
        ls="",
        label=f"from circlefit - {uuid}",
    )

# Axis formatting
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(5e2, 1e10)

ax.set_title("PSD of frequency time traces")
ax.set_xlabel(r"$f\ (\mathrm{Hz})$", fontsize=30)
ax.set_ylabel(r"PSD $S(f)\ (\mathrm{\frac{Hz^2}{Hz}})$", fontsize=30)

ax.tick_params(axis="both", labelsize=28)
ax.legend(markerscale=10, fontsize=30)

plt.show()

## Noise Spectral Density of Frequency-Time Traces

### NSD Comparison Configuration

In [ ]:
UUID_TO_STYLE = {
    "S0BI8H": {
        "color": "#00c0f3",
        "marker": "s",
        "label": r"$L_s = 182.5\,\mathrm{pH/sq}$",
        "linestyle": "None",
        "linewidth": 1,
        "markersize": 0.3,
    },
    "S2XXMP": {
        "color": "#9f58db",
        "marker": "s",
        "label": r"$L_s = 260\,\mathrm{pH/sq}$",
        "linestyle": "None",
        "linewidth": 1,
        "markersize": 0.3,
    },
    "S911RA": {
        "color": "#00A76D",
        "marker": "s",
        "label": r"$L_s = 670\,\mathrm{pH/sq}$",
        "linestyle": "None",
        "linewidth": 1,
        "markersize": 0.3,
    },
    "S9SL0J": {
        "color": "#F39C12",
        "marker": "s",
        "label": r"$L_s = 530\,\mathrm{pH/sq}$",
        "linestyle": "None",
        "linewidth": 1,
        "markersize": 0.3,
    },
    # Add this only if present in the loaded data:
    # "S9SL0Z": {
    #     "color": "#E74C3C",
    #     "marker": "s",
    #     "label": r"$L_s = ...\,\mathrm{pH/sq}$",
    #     "linestyle": "None",
    #     "linewidth": 1,
    #     "markersize": 0.3,
    # },
}

PROCESSED_FILES = [
    "Time_trace_S911RA_S2XXMP_S0BI8H.npz",
    "Time_trace_S9SL0J_S9SL0Z.npz",
]

FIG_SIZE = (4.2, 2.1)
Y_LIMITS = (1e1, 4e4)


### Load and Combine Processed Resonator Data

In [ ]:
resonator_data = {}

for file_path in PROCESSED_FILES:
    loaded_data = np.load(file_path, allow_pickle=True)
    resonator_data.update(loaded_data)

### Plot NSD for Selected Resonators

In [ ]:
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.set_position([0.15, 0.15, 0.8, 0.8])

for uuid in resonator_data:
    if uuid not in UUID_TO_STYLE:
        print(f"Warning: no plotting style defined for {uuid}. Skipping.")
        continue

    dataset = resonator_data[uuid].item()
    f_ts = dataset["frequency_time_series"]
    t_per_trace = np.mean(np.diff(dataset["time_series"]))

    # Compute average NSD
    fft_f, nsd_avg, _ = NSD_avg(f_ts, t_per_trace)

    style = UUID_TO_STYLE[uuid]

    ax.plot(
        fft_f,
        nsd_avg,
        marker=style["marker"],
        linestyle=style["linestyle"],
        linewidth=style["linewidth"],
        color=style["color"],
        label=style["label"],
        markersize=style["markersize"],
    )

# Axis formatting
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(*Y_LIMITS)

# NOTE: frequency axis from PSD/NSD is typically in Hz
ax.set_xlabel(r'$\mathit{f}\ (\mathrm{Hz})$', fontsize=8, labelpad=2)
ax.set_ylabel(r'$\mathrm{NSD}\ (\mathrm{Hz/\sqrt{Hz}})$', fontsize=8, labelpad=2)

ax.tick_params(axis='both', which='major', length=2, labelsize=8, pad=1)
ax.tick_params(axis='both', which='minor', length=1.3, width=0.7)

# Optional legend
# ax.legend(loc="lower left", fontsize=8, handletextpad=0.5, labelspacing=0.5, frameon=False)

plt.show()

# Resonance-Frequency Summary and Impedance Comparison

## Combined Figure Configuration

In [ ]:
# Measured resonance frequencies [GHz]
RESONANCE_FREQUENCIES_GHZ = [
    5.955, 5.98, 8.87, 8.9, 6.43, 6.54, 7.02, 7.10,
    4.688, 4.769, 5.009, 5.110, 5.783, 5.869,
    5.70, 5.81, 6.03, 6.13, 6.80, 6.92, 7.39, 7.49,
]

# Corresponding sheet inductance labels
SHEET_INDUCTANCES = (
    ["180", "180"]
    + ["260"] * 6
    + ["670"] * 6
    + ["600"] * 4
    + ["530"] * 4
)

# Color mapping for each sheet inductance
LS_COLORS = {
    "180": "#00c0f3",
    "260": "#9f58db",
    "670": "#00A76D",
    "600": "#F7931D",
    "530": "#FFCD05",
}

# Scatter-plot data: impedance vs resonance frequency
frequency_1 = np.array([4.688, 5.009, 5.783])   # GHz
impedance_1 = np.array([127, 113, 104])         # kOhm

frequency_2 = np.array([8.87, 6.43, 7.02])      # GHz
impedance_2 = np.array([89, 86, 81])            # kOhm

frequency_3 = np.array([5.955])                 # GHz
impedance_3 = np.array([74])                    # kOhm

frequency_4 = np.array([5.70, 6.03])            # GHz
impedance_4 = np.array([126, 112])              # kOhm

frequency_5 = np.array([6.80, 7.39])            # GHz
impedance_5 = np.array([120, 111])              # kOhm

UNIQUE_LS = sorted(set(SHEET_INDUCTANCES), key=lambda x: int(x))

## Generate Combined Figure

In [ ]:
fig = plt.figure(figsize=(3.00, len(UNIQUE_LS) * 0.05 + 1.2))
gs = gridspec.GridSpec(
    len(UNIQUE_LS) + 1,
    1,
    height_ratios=[0.3] * len(UNIQUE_LS) + [1.2],
    hspace=0,
)

# ------------------------------------------------------------
# Top panel: stacked resonance-frequency overview
# ------------------------------------------------------------
for i, ls_value in enumerate(UNIQUE_LS):
    ax = plt.subplot(gs[i])

    for j, fr in enumerate(RESONANCE_FREQUENCIES_GHZ):
        if SHEET_INDUCTANCES[j] == ls_value:
            ax.axvline(fr, c=LS_COLORS[ls_value], linewidth=0.7, alpha=0.9)

    # Single y-tick showing the corresponding Ls value
    ax.set_yticks([0.5])
    ax.set_yticklabels([ls_value], fontsize=8)
    ax.tick_params(
        axis="y",
        which="both",
        length=2,
        labelsize=8,
        width=1,
        pad=2,
        colors="black",
    )

    # Hide x-axis ticks for all upper subplots
    if i < len(UNIQUE_LS) - 1:
        ax.set_xticks([])
        ax.set_xticklabels([])

    ax.set_xlim(4.5, 9.0)
    ax.tick_params(
        axis="x",
        which="both",
        bottom=False,
        top=False,
        labelbottom=False,
        length=2,
        labelsize=8,
        width=1,
        pad=2,
    )

    # Match subplot border style
    ax.spines["top"].set_linewidth(1)
    ax.spines["bottom"].set_linewidth(1)
    ax.spines["left"].set_linewidth(1)
    ax.spines["right"].set_linewidth(1)

# ------------------------------------------------------------
# Bottom panel: impedance vs frequency scatter plot
# ------------------------------------------------------------
ax_scatter = plt.subplot(gs[-1])

ax_scatter.scatter(frequency_1, impedance_1, s=8, c="#00A76D", marker="o", label="Group 1")
ax_scatter.scatter(frequency_2, impedance_2, s=8, c="#9f58db", marker="o", label="Group 2")
ax_scatter.scatter(frequency_3, impedance_3, s=8, c="#00c0f3", marker="o", label="Group 3")
ax_scatter.scatter(frequency_4, impedance_4, s=8, c="#F7931D", marker="o", label="Group 4")
ax_scatter.scatter(frequency_5, impedance_5, s=8, c="#FFCD05", marker="o", label="Group 5")

ax_scatter.set_xlabel(r"$f$ (GHz)", fontsize=8, labelpad=2, color="black")
ax_scatter.set_ylabel(r"$Z$ (k$\Omega$)", fontsize=8, labelpad=2, color="black")

# X-axis formatting
x_start, x_end = 4.5, 9.0
x_ticks = np.arange(4.5, 9.5, 0.5)

ax_scatter.set_xlim(x_start, x_end)
ax_scatter.set_xticks(x_ticks)

def custom_formatter(val, pos):
    if float(val).is_integer():
        return f"{int(val)}"
    return f"{val:.1f}"

ax_scatter.xaxis.set_major_formatter(mticker.FuncFormatter(custom_formatter))

# Y-axis formatting
y_start, y_end = 65, 135
y_ticks = np.arange(60, 140, 20)

ax_scatter.set_ylim(y_start, y_end)
ax_scatter.set_yticks(y_ticks)
ax_scatter.set_yticklabels([f"{y:.0f}" for y in y_ticks], fontsize=8)

ax_scatter.tick_params(axis="both", which="major", length=2, labelsize=8, width=1, pad=2)

# Match border style with upper panels
ax_scatter.spines["top"].set_linewidth(1)
ax_scatter.spines["bottom"].set_linewidth(1)
ax_scatter.spines["left"].set_linewidth(1)
ax_scatter.spines["right"].set_linewidth(1)

# Global layout adjustment
plt.subplots_adjust(hspace=0, wspace=0, bottom=0.05, top=0.98, left=0.15, right=0.95)

# Optional export
# save_path = r"C:\PhD_project\Paper\Hight_Impedance_Resonators\figs\3\combined_sticky1.png"
# fig.savefig(save_path, dpi=1000, bbox_inches="tight", transparent=True)

plt.show()

# Impedance Calculation

In [ ]:
import numpy as np

def calculate_impedance(L_microH, F_GHz):
    """
    Calculate the impedance Z in kilohms.
    
    Parameters:
    L_microH: Inductance in microhenries
    F_GHz: Frequency in gigahertz
    
    Returns:
    Z_kohms: Impedance in kilohms
    """
    L = L_microH * 1e-6  # Convert microhenries to henries
    F = F_GHz * 1e9      # Convert gigahertz to hertz
    omega = 2 * np.pi * F  # Angular frequency
    Z = omega * L         # Impedance in ohms
    Z_kohms = Z / 1e3     # Convert ohms to kilohms
    return Z_kohms

# Example usage:
L_microH = 2.5  # Inductance in microhenries
F_GHz = 4.6     # Frequency in gigahertz
Z_kohms = calculate_impedance(L_microH, F_GHz)
print(f"Lω = {Z_kohms:.3f} kΩ")
